In [1]:
import pandas as pd
import numpy as np
import os
import pickle
import bambi as bmb
import matplotlib.pyplot as plt
from s0_fun_base import XY_sur_compu, XY_grw_f_compu, XY_grw_nf_compu, XY_fec_compu, XY_flow_compu, XY_grw_compu
import statsmodels.api as sm
import statsmodels.formula.api as smf
import gpflow
# simulating true populations
from s0_fun_IBMs import IBM_1step_glm_mle, IBM_1step_gp, popu_structure

In [2]:
# This file is used to pre-process 'raw' population datasets... 
reproc_data_mode = 'OFF'
Time = 3 # the number of years
re_calcu_TRUEmodels = 'OFF'
re_generate_MLE_true_population = 'OFF'
re_MCMC = 'OFF'
re_calcu_glm_mlemle_models = 'OFF'
re_calcu_gp_mlemle_models = 'OFF'

In [3]:
if reproc_data_mode == 'ON':    
    print("\n\n Re-processing 'raw' population datasets... \n\n")
    for i in range(Time):
        if i == 0:
            data = pd.read_csv("true_popu/control_89.csv")
        elif i == 1:
            data = pd.read_csv("true_popu/control_910.csv")
        elif i == 2:
            data = pd.read_csv("true_popu/control_1011.csv")

        data = data.drop(data.columns[0], axis="columns")
        data = data.sort_values(by=['size']).reset_index(drop=True)

        # some inidividuals' size are zero, whcih mighe be sampling errors.
        data = data.drop(np.where(data['size'] == 0)[0], axis=0).reset_index(drop=True)
        data = data.drop(np.where(data['sizeNext'] == 0)[0], axis=0).reset_index(drop=True)
        data['size'] = np.log(data['size'])
        data['sizeNext'] = np.log(data['sizeNext'])
        # classify individuals into several different age classes for latter analysis
        # the largest value of age in dataset 2008-2009 is 12, we set 13 as an obsorbing age.
        data['age'][data['age'] > 13] = 13
        data['ageNext'][data['ageNext'] > 13] = 13
        pickle.dump(data, open(file = os.getcwd()+f"/true_popu/popu_dataset{i}.pkl", mode="wb"))

In [4]:
if reproc_data_mode == 'ON': 
        for i in range(Time):
                globals()[f'popu_dataset{i}'] = pickle.load(open(file = os.getcwd()+f"/true_popu/popu_dataset{i}.pkl", mode="rb"))
        p = {i: globals()[f'popu_dataset{i}'].copy() for i in range(Time)}
        popu_whole = pd.concat([p[i] for i in range(Time)]).sort_values(by=['size']).reset_index(drop=True); del(p)
        pickle.dump(popu_whole, open(file = os.getcwd()+f"/true_popu/popu_whole.pkl", mode="wb"))
else:
        popu_whole = pickle.load(open(file = os.getcwd()+f"/true_popu/popu_whole.pkl", mode="rb"))

In [5]:
popu_whole

,size,flow,sizeNext,age,ageNext,surv,fec
0,0.0,NaN,NaN,1.0,NaN,0.0,0.0
1,0.0,NaN,1.386294,1.0,2.0,1.0,0.0
2,0.0,NaN,0.000000,1.0,2.0,1.0,0.0
3,0.0,NaN,1.098612,2.0,3.0,1.0,0.0
4,0.0,NaN,NaN,1.0,NaN,0.0,0.0
...,...,...,...,...,...,...,...
439,NaN,NaN,0.000000,NaN,1.0,NaN,NaN
440,NaN,NaN,1.945910,NaN,1.0,NaN,NaN
441,NaN,NaN,0.000000,NaN,1.0,NaN,NaN
442,NaN,NaN,1.386294,NaN,1.0,NaN,NaN


In [6]:
if re_calcu_TRUEmodels == 'ON':
    # calculate Xs and ys
    X_sur, y_sur = XY_sur_compu(popu_whole)
    X_fec, y_fec = XY_fec_compu(popu_whole)
    X_flow, y_flow = XY_flow_compu(popu_whole)
    X_grw_f, y_grw_f = XY_grw_f_compu(popu_whole)
    X_grw_nf, y_grw_nf = XY_grw_nf_compu(popu_whole)
    
    # Calculating MLEs for GLMs
    model_sur_glm = smf.glm("sur ~ size", data=pd.concat((X_sur, y_sur), axis=1), family=sm.families.Binomial()).fit()
    model_fec_glm = smf.glm("fec ~ size", data=pd.concat((X_fec, y_fec), axis=1), family=sm.families.Binomial()).fit()
    model_flow_glm = smf.glm("flow ~ size", data=pd.concat((X_flow, y_flow), axis=1), family=sm.families.Poisson()).fit()
    model_grw_f_glm = smf.ols("sizeNext ~ size", data=pd.concat((X_grw_f, y_grw_f), axis=1)).fit()
    model_grw_nf_glm = smf.ols("sizeNext ~ size", data=pd.concat((X_grw_nf, y_grw_nf), axis=1)).fit()

    # Calculating MLEs for GPMs
    # 1 - survive 
    m_sur1 = gpflow.models.VGP((X_sur, y_sur), kernel=gpflow.kernels.RBF(), likelihood=gpflow.likelihoods.Bernoulli())
    opt_sur = gpflow.optimizers.Scipy()
    opt_sur.minimize(m_sur1.training_loss, variables=m_sur1.trainable_variables, options=dict(maxiter=10000))

    # 2 - growth: if we conserding "fec" effect
    # growth model fitting for breeders (fec == 1)
    m_grw_f = gpflow.models.GPR(data=(X_grw_f, y_grw_f), kernel=gpflow.kernels.RBF())
    opt_grw_f = gpflow.optimizers.Scipy()
    opt_grw_f.minimize(m_grw_f.training_loss, variables=m_grw_f.trainable_variables, options=dict(maxiter=10000))

    # 3 - growth model fitting for the remainings (fec == 0)
    m_grw_nf = gpflow.models.GPR(data=(X_grw_nf, y_grw_nf), kernel=gpflow.kernels.RBF())
    opt_grw_nf = gpflow.optimizers.Scipy()
    opt_grw_nf.minimize(m_grw_nf.training_loss, variables=m_grw_nf.trainable_variables, options=dict(maxiter=10000))

    # 4 - fec
    m_fec = gpflow.models.VGP((X_fec, y_fec), kernel=gpflow.kernels.RBF(), likelihood=gpflow.likelihoods.Bernoulli())
    opt_fec = gpflow.optimizers.Scipy()
    opt_fec.minimize(m_fec.training_loss, variables=m_fec.trainable_variables, options=dict(maxiter=10000))

    # 5 - number of flowering stalks: Poisson
    m_flow_poi = gpflow.models.VGP((X_flow, y_flow), kernel=gpflow.kernels.RBF(), likelihood=gpflow.likelihoods.Poisson())
    opt_flow_poi = gpflow.optimizers.Scipy()
    opt_flow_poi.minimize(m_flow_poi.training_loss, variables=m_flow_poi.trainable_variables, options=dict(maxiter=10000))

    # recruit size (indp with the parent size)
    index_recr = np.isnan(popu_whole["size"])
    sizeNext_recr = popu_whole["sizeNext"][index_recr].to_numpy()    
    mean_log_recr = np.mean(sizeNext_recr)
    var_log_recr = np.var(sizeNext_recr)
    v = var_log_recr * sizeNext_recr.shape[0] / (sizeNext_recr.shape[0] - 1)
    # for gamma distribution
    alpha = np.power(mean_log_recr, 2) / v
    beta = mean_log_recr / v
    recruit_estab = sum(np.isnan(popu_whole["size"])) / popu_whole["flow"].sum()

    glm_mle_true_models = {'m_sur': model_sur_glm,
                    'm_fec': model_fec_glm,
                    'm_flow_poi': model_flow_glm,
                    'm_grw_f': model_grw_f_glm,
                    'm_grw_nf': model_grw_nf_glm,
                    'alpha': alpha,
                    'beta': beta,
                    'recruit_p': recruit_estab
                    }

    gp_mle_true_models = {'m_sur': m_sur1,
                    'm_fec': m_fec,
                    'm_flow_poi': m_flow_poi,
                    'm_grw_f': m_grw_f,
                    'm_grw_nf': m_grw_nf,
                    'alpha': alpha,
                    'beta': beta,
                    'recruit_p': recruit_estab
                    }
    
    pickle.dump(glm_mle_true_models, open(file = os.getcwd()+f"/true_popu/glm_mle_true_models.pkl", mode="wb"))
    pickle.dump(gp_mle_true_models, open(file = os.getcwd()+f"/true_popu/gp_mle_true_models.pkl", mode="wb"))
else:
    gp_mle_true_models = pickle.load(open(file = os.getcwd()+f"/true_popu/gp_mle_true_models.pkl", mode="rb"))
    glm_mle_true_models = pickle.load(open(file = os.getcwd()+f"/true_popu/glm_mle_true_models.pkl", mode="rb"))

In [11]:
if re_generate_MLE_true_population == 'ON':
    # for glm
    data_simu = popu_whole.copy()
    for _ in range(10):
        z = popu_structure(data_simu) 
        data_simu = IBM_1step_glm_mle(zt=z[0], age=z[1], models=glm_mle_true_models) 
    pickle.dump(data_simu, open(file = os.getcwd()+f"/true_popu/glm_mle_true_population/glm_mle_true_population.pkl", mode="wb"))

    # for gp models
    data_simu = popu_whole.copy()
    for _ in range(10):
        z = popu_structure(data_simu) 
        data_simu = IBM_1step_gp(zt=z[0], age=z[1], models=gp_mle_true_models) 

    pickle.dump(data_simu, open(file = os.getcwd()+f"/true_popu/gp_mle_true_population/gp_mle_true_population.pkl", mode="wb")) 

else:
    glm_mle_true_population = pickle.load(open(file = os.getcwd()+f"/true_popu/glm_mle_true_population/glm_mle_true_population.pkl", mode="rb"))
    gp_mle_true_population = pickle.load(open(file = os.getcwd()+f"/true_popu/gp_mle_true_population/gp_mle_true_population.pkl", mode="rb"))  

In [12]:
glm_mle_true_population

,size,sizeNext,fec,flow,surv,age,ageNext
0,1.992937,2.634334,1.0,2.0,1.0,11.0,12.0
1,1.777610,2.835917,0.0,NaN,1.0,11.0,12.0
2,1.864109,3.032237,0.0,NaN,1.0,11.0,12.0
3,1.772487,1.210048,0.0,NaN,1.0,13.0,13.0
4,3.160306,NaN,1.0,6.0,0.0,11.0,NaN
...,...,...,...,...,...,...,...
557,NaN,1.447285,NaN,NaN,NaN,NaN,1.0
558,NaN,0.339528,NaN,NaN,NaN,NaN,1.0
559,NaN,0.167697,NaN,NaN,NaN,NaN,1.0
560,NaN,0.653392,NaN,NaN,NaN,NaN,1.0


In [13]:
gp_mle_true_population

,size,sizeNext,fec,flow,surv,age,ageNext
0,2.479433,2.838933,0.0,NaN,1.0,12.0,13.0
1,2.094642,NaN,1.0,2.0,0.0,13.0,NaN
2,3.925451,3.933102,1.0,7.0,1.0,12.0,13.0
3,4.415770,NaN,1.0,14.0,0.0,13.0,NaN
4,2.750754,NaN,1.0,7.0,0.0,12.0,NaN
...,...,...,...,...,...,...,...
296,NaN,0.397035,NaN,NaN,NaN,NaN,1.0
297,NaN,0.399597,NaN,NaN,NaN,NaN,1.0
298,NaN,1.490389,NaN,NaN,NaN,NaN,1.0
299,NaN,1.336871,NaN,NaN,NaN,NaN,1.0


In [15]:
if re_calcu_glm_mlemle_models == 'ON':
    # Calculate MLEs on simulated 'TURE' datsets

    # for glm generated datasets
    X_sur, y_sur = XY_sur_compu(glm_mle_true_population)
    X_fec, y_fec = XY_fec_compu(glm_mle_true_population)
    X_flow, y_flow = XY_flow_compu(glm_mle_true_population)
    X_grw_f, y_grw_f = XY_grw_f_compu(glm_mle_true_population)
    X_grw_nf, y_grw_nf = XY_grw_nf_compu(glm_mle_true_population)
    X_grw, y_grw = XY_grw_compu(glm_mle_true_population)
    
    # Calculating MLEs for GLMs
    model_sur_glm = smf.glm("sur ~ size", data=pd.concat((X_sur, y_sur), axis=1), family=sm.families.Binomial()).fit()
    model_fec_glm = smf.glm("fec ~ size", data=pd.concat((X_fec, y_fec), axis=1), family=sm.families.Binomial()).fit()
    model_flow_glm = smf.glm("flow ~ size", data=pd.concat((X_flow, y_flow), axis=1), family=sm.families.Poisson()).fit()
    model_grw_f_glm = smf.ols("sizeNext ~ size", data=pd.concat((X_grw_f, y_grw_f), axis=1)).fit()
    model_grw_nf_glm = smf.ols("sizeNext ~ size", data=pd.concat((X_grw_nf, y_grw_nf), axis=1)).fit()
    model_grw_glm = smf.ols("sizeNext ~ size", data=pd.concat((X_grw, y_grw), axis=1)).fit()

    # Calculating MLEs for GPMs
    # 1 - survive 
    m_sur1 = gpflow.models.VGP((X_sur, y_sur), kernel=gpflow.kernels.RBF(), likelihood=gpflow.likelihoods.Bernoulli())
    opt_sur = gpflow.optimizers.Scipy()
    opt_sur.minimize(m_sur1.training_loss, variables=m_sur1.trainable_variables, options=dict(maxiter=10000))

    # 2.1 - growth: if we conserding "fec" effect
    # growth model fitting for breeders (fec == 1)
    m_grw_f = gpflow.models.GPR(data=(X_grw_f, y_grw_f), kernel=gpflow.kernels.RBF())
    opt_grw_f = gpflow.optimizers.Scipy()
    opt_grw_f.minimize(m_grw_f.training_loss, variables=m_grw_f.trainable_variables, options=dict(maxiter=10000))

    # 2.2 - growth model fitting for the remainings (fec == 0)
    m_grw_nf = gpflow.models.GPR(data=(X_grw_nf, y_grw_nf), kernel=gpflow.kernels.RBF())
    opt_grw_nf = gpflow.optimizers.Scipy()
    opt_grw_nf.minimize(m_grw_nf.training_loss, variables=m_grw_nf.trainable_variables, options=dict(maxiter=10000))

    # 3 - growth model fitting for all
    m_grw = gpflow.models.GPR(data=(X_grw, y_grw), kernel=gpflow.kernels.RBF())
    opt_grw = gpflow.optimizers.Scipy()
    opt_grw.minimize(m_grw.training_loss, variables=m_grw.trainable_variables, options=dict(maxiter=10000))

    # 4 - fec
    m_fec = gpflow.models.VGP((X_fec, y_fec), kernel=gpflow.kernels.RBF(), likelihood=gpflow.likelihoods.Bernoulli())
    opt_fec = gpflow.optimizers.Scipy()
    opt_fec.minimize(m_fec.training_loss, variables=m_fec.trainable_variables, options=dict(maxiter=10000))

    # 5 - number of flowering stalks: Poisson
    m_flow_poi = gpflow.models.VGP((X_flow, y_flow), kernel=gpflow.kernels.RBF(), likelihood=gpflow.likelihoods.Poisson())
    opt_flow_poi = gpflow.optimizers.Scipy()
    opt_flow_poi.minimize(m_flow_poi.training_loss, variables=m_flow_poi.trainable_variables, options=dict(maxiter=10000))

    # recruit size (indp with the parent size)
    index_recr = np.isnan(glm_mle_true_population["size"])
    sizeNext_recr = glm_mle_true_population["sizeNext"][index_recr].to_numpy()    
    mean_log_recr = np.mean(sizeNext_recr)
    var_log_recr = np.var(sizeNext_recr)
    v = var_log_recr * sizeNext_recr.shape[0] / (sizeNext_recr.shape[0] - 1)
    # for gamma distribution
    alpha = np.power(mean_log_recr, 2) / v
    beta = mean_log_recr / v
    recruit_estab = sum(np.isnan(glm_mle_true_population["size"])) / glm_mle_true_population["flow"].sum()

    glm_mle_mle_models = {'m_sur': model_sur_glm,
                    'm_fec': model_fec_glm,
                    'm_flow_poi': model_flow_glm,
                    'm_grw_f': model_grw_f_glm,
                    'm_grw_nf': model_grw_nf_glm,
                    'm_grw': model_grw_glm,
                    'alpha': alpha,
                    'beta': beta,
                    'recruit_p': recruit_estab
                    }

    gp_mle_mle_models = {'m_sur': m_sur1,
                    'm_fec': m_fec,
                    'm_flow_poi': m_flow_poi,
                    'm_grw_f': m_grw_f,
                    'm_grw_nf': m_grw_nf,
                    'm_grw': m_grw,
                    'alpha': alpha,
                    'beta': beta,
                    'recruit_p': recruit_estab
                    }
    
    pickle.dump(glm_mle_mle_models, open(file = os.getcwd()+f"/true_popu/glm_mle_true_population/glm_glmmle_mle_models.pkl", mode="wb"))
    pickle.dump(gp_mle_mle_models, open(file = os.getcwd()+f"/true_popu/glm_mle_true_population/gp_glmmle_mle_models.pkl", mode="wb"))
else:
    glm_glmmle_mle_models = pickle.load(open(file = os.getcwd()+f"/true_popu/glm_mle_true_population/glm_glmmle_mle_models.pkl", mode="rb"))
    gp_glmmle_mle_models = pickle.load(open(file = os.getcwd()+f"/true_popu/glm_mle_true_population/gp_glmmle_mle_models.pkl", mode="rb"))

In [15]:
if re_calcu_gp_mlemle_models == 'ON':
    # Calculate MLEs on simulated 'TURE' datsets

    # for gp generated datasets
    X_sur, y_sur = XY_sur_compu(gp_mle_true_population)
    X_fec, y_fec = XY_fec_compu(gp_mle_true_population)
    X_flow, y_flow = XY_flow_compu(gp_mle_true_population)
    X_grw_f, y_grw_f = XY_grw_f_compu(gp_mle_true_population)
    X_grw_nf, y_grw_nf = XY_grw_nf_compu(gp_mle_true_population)
    X_grw, y_grw = XY_grw_compu(gp_mle_true_population)
    
    # Calculating MLEs for GLMs
    model_sur_glm = smf.glm("sur ~ size", data=pd.concat((X_sur, y_sur), axis=1), family=sm.families.Binomial()).fit()
    model_fec_glm = smf.glm("fec ~ size", data=pd.concat((X_fec, y_fec), axis=1), family=sm.families.Binomial()).fit()
    model_flow_glm = smf.glm("flow ~ size", data=pd.concat((X_flow, y_flow), axis=1), family=sm.families.Poisson()).fit()
    model_grw_f_glm = smf.ols("sizeNext ~ size", data=pd.concat((X_grw_f, y_grw_f), axis=1)).fit()
    model_grw_nf_glm = smf.ols("sizeNext ~ size", data=pd.concat((X_grw_nf, y_grw_nf), axis=1)).fit()
    model_grw_glm = smf.ols("sizeNext ~ size", data=pd.concat((X_grw, y_grw), axis=1)).fit()

    # Calculating MLEs for GPMs
    # 1 - survive 
    m_sur1 = gpflow.models.VGP((X_sur, y_sur), kernel=gpflow.kernels.RBF(), likelihood=gpflow.likelihoods.Bernoulli())
    opt_sur = gpflow.optimizers.Scipy()
    opt_sur.minimize(m_sur1.training_loss, variables=m_sur1.trainable_variables, options=dict(maxiter=10000))

    # 2.1 - growth: if we conserding "fec" effect
    # growth model fitting for breeders (fec == 1)
    m_grw_f = gpflow.models.GPR(data=(X_grw_f, y_grw_f), kernel=gpflow.kernels.RBF())
    opt_grw_f = gpflow.optimizers.Scipy()
    opt_grw_f.minimize(m_grw_f.training_loss, variables=m_grw_f.trainable_variables, options=dict(maxiter=10000))

    # 2.2 - growth model fitting for the remainings (fec == 0)
    m_grw_nf = gpflow.models.GPR(data=(X_grw_nf, y_grw_nf), kernel=gpflow.kernels.RBF())
    opt_grw_nf = gpflow.optimizers.Scipy()
    opt_grw_nf.minimize(m_grw_nf.training_loss, variables=m_grw_nf.trainable_variables, options=dict(maxiter=10000))

    # 3 - growth model fitting for all
    m_grw = gpflow.models.GPR(data=(X_grw, y_grw), kernel=gpflow.kernels.RBF())
    opt_grw = gpflow.optimizers.Scipy()
    opt_grw.minimize(m_grw.training_loss, variables=m_grw.trainable_variables, options=dict(maxiter=10000))

    # 4 - fec
    m_fec = gpflow.models.VGP((X_fec, y_fec), kernel=gpflow.kernels.RBF(), likelihood=gpflow.likelihoods.Bernoulli())
    opt_fec = gpflow.optimizers.Scipy()
    opt_fec.minimize(m_fec.training_loss, variables=m_fec.trainable_variables, options=dict(maxiter=10000))

    # 5 - number of flowering stalks: Poisson
    m_flow_poi = gpflow.models.VGP((X_flow, y_flow), kernel=gpflow.kernels.RBF(), likelihood=gpflow.likelihoods.Poisson())
    opt_flow_poi = gpflow.optimizers.Scipy()
    opt_flow_poi.minimize(m_flow_poi.training_loss, variables=m_flow_poi.trainable_variables, options=dict(maxiter=10000))

    # recruit size (indp with the parent size)
    index_recr = np.isnan(gp_mle_true_population["size"])
    sizeNext_recr = gp_mle_true_population["sizeNext"][index_recr].to_numpy()    
    mean_log_recr = np.mean(sizeNext_recr)
    var_log_recr = np.var(sizeNext_recr)
    v = var_log_recr * sizeNext_recr.shape[0] / (sizeNext_recr.shape[0] - 1)
    # for gamma distribution
    alpha = np.power(mean_log_recr, 2) / v
    beta = mean_log_recr / v
    recruit_estab = sum(np.isnan(gp_mle_true_population["size"])) / gp_mle_true_population["flow"].sum()

    glm_mle_mle_models = {'m_sur': model_sur_glm,
                    'm_fec': model_fec_glm,
                    'm_flow_poi': model_flow_glm,
                    'm_grw_f': model_grw_f_glm,
                    'm_grw_nf': model_grw_nf_glm,
                    'm_grw': model_grw_glm,
                    'alpha': alpha,
                    'beta': beta,
                    'recruit_p': recruit_estab
                    }

    gp_mle_mle_models = {'m_sur': m_sur1,
                    'm_fec': m_fec,
                    'm_flow_poi': m_flow_poi,
                    'm_grw_f': m_grw_f,
                    'm_grw_nf': m_grw_nf,
                    'm_grw': m_grw,
                    'alpha': alpha,
                    'beta': beta,
                    'recruit_p': recruit_estab
                    }
    
    pickle.dump(glm_mle_mle_models, open(file = os.getcwd()+f"/true_popu/gp_mle_true_population/glm_gpmle_mle_models.pkl", mode="wb"))
    pickle.dump(gp_mle_mle_models, open(file = os.getcwd()+f"/true_popu/gp_mle_true_population/gp_gpmle_mle_models.pkl", mode="wb"))
else:
    glm_gpmle_mle_models = pickle.load(open(file = os.getcwd()+f"/true_popu/gp_mle_true_population/glm_gpmle_mle_models.pkl", mode="rb"))
    gp_gpmle_mle_models = pickle.load(open(file = os.getcwd()+f"/true_popu/gp_mle_true_population/gp_gpmle_mle_models.pkl", mode="rb"))

In [19]:
if re_MCMC == 'ON':
    # MCMC for GLMs
    # for the 1st datastes glm_mle_true_population with sep setting (seprate models for breeders and non-breeders).
    # grw_f
    df_grw_f = XY_grw_f_compu(glm_mle_true_population)
    model_grw_f = bmb.Model("sizeNext ~ size", pd.concat(df_grw_f, axis=1))
    trace_grw_f = model_grw_f.fit(draws=5000, tune=10000, discard_tuned_samples=True, chains=1, progressbar=False)
    pickle.dump(trace_grw_f, open(file = os.getcwd()+f"/true_popu/glm_mle_true_population/MCMC/glm/sep/trace_grw_f.pkl", mode="wb"))

    # grw_nf
    df_grw_nf = XY_grw_nf_compu(glm_mle_true_population)
    model_grw_nf = bmb.Model("sizeNext ~ size", pd.concat(df_grw_nf, axis=1))
    trace_grw_nf = model_grw_nf.fit(draws=5000, tune=10000, discard_tuned_samples=True, chains=1, progressbar=False)
    pickle.dump(trace_grw_nf, open(file = os.getcwd()+f"/true_popu/glm_mle_true_population/MCMC/glm/sep/trace_grw_nf.pkl", mode="wb"))

    # sur
    df_sur = XY_sur_compu(glm_mle_true_population)
    model_sur = bmb.Model("sur ~ size", pd.concat(df_sur, axis=1), family='bernoulli')
    trace_sur = model_sur.fit(draws=5000, tune=10000, discard_tuned_samples=True, chains=1, progressbar=False)
    pickle.dump(trace_sur, open(file = os.getcwd()+f"/true_popu/glm_mle_true_population/MCMC/glm/sep/trace_sur.pkl", mode="wb"))

    # fec
    df_fec = XY_fec_compu(glm_mle_true_population)
    model_fec = bmb.Model("fec ~ size", pd.concat(df_fec, axis=1), family='bernoulli')
    trace_fec = model_fec.fit(draws=5000, tune=10000, discard_tuned_samples=True, chains=1, progressbar=False)
    pickle.dump(trace_fec, open(file = os.getcwd()+f"/true_popu/glm_mle_true_population/MCMC/glm/sep/trace_fec.pkl", mode="wb"))

    # flow
    df_flow = XY_flow_compu(glm_mle_true_population)
    model_flow = bmb.Model("flow ~ size", pd.concat(df_flow, axis=1), family='poisson')
    trace_flow = model_flow.fit(draws=5000, tune=10000, discard_tuned_samples=True, chains=1, progressbar=False)
    pickle.dump(trace_flow, open(file = os.getcwd()+f"/true_popu/glm_mle_true_population/MCMC/glm/sep/trace_flow.pkl", mode="wb"))

    # for the 1st datastes glm_mle_true_population with non-sep setting (seprate models for breeders and non-breeders).
    # grw, there is no needs to calculate the otehr vital rates.
    df_grw = XY_grw_compu(glm_mle_true_population)
    model_grw = bmb.Model("sizeNext ~ size", pd.concat(df_grw, axis=1))
    trace_grw = model_grw.fit(draws=5000, tune=10000, discard_tuned_samples=True, chains=1, progressbar=False)
    pickle.dump(trace_grw, open(file = os.getcwd()+f"/true_popu/glm_mle_true_population/MCMC/glm/nonsep/trace_grw.pkl", mode="wb"))



    # MCMC for GPs
    # for the 2nd datastes gp_mle_true_population with sep setting (seprate models for breeders and non-breeders).
    # grw_f
    df_grw_f = XY_grw_f_compu(gp_mle_true_population)
    model_grw_f = bmb.Model("sizeNext ~ size", pd.concat(df_grw_f, axis=1))
    trace_grw_f = model_grw_f.fit(draws=5000, tune=10000, discard_tuned_samples=True, chains=1, progressbar=False)
    pickle.dump(trace_grw_f, open(file = os.getcwd()+f"/true_popu/gp_mle_true_population/MCMC/glm/sep/trace_grw_f.pkl", mode="wb"))

    # grw_nf
    df_grw_nf = XY_grw_nf_compu(gp_mle_true_population)
    model_grw_nf = bmb.Model("sizeNext ~ size", pd.concat(df_grw_nf, axis=1))
    trace_grw_nf = model_grw_nf.fit(draws=5000, tune=10000, discard_tuned_samples=True, chains=1, progressbar=False)
    pickle.dump(trace_grw_nf, open(file = os.getcwd()+f"/true_popu/gp_mle_true_population/MCMC/glm/sep/trace_grw_nf.pkl", mode="wb"))

    # sur
    df_sur = XY_sur_compu(gp_mle_true_population)
    model_sur = bmb.Model("sur ~ size", pd.concat(df_sur, axis=1), family='bernoulli')
    trace_sur = model_sur.fit(draws=5000, tune=10000, discard_tuned_samples=True, chains=1, progressbar=False)
    pickle.dump(trace_sur, open(file = os.getcwd()+f"/true_popu/gp_mle_true_population/MCMC/glm/sep/trace_sur.pkl", mode="wb"))

    # fec
    df_fec = XY_fec_compu(gp_mle_true_population)
    model_fec = bmb.Model("fec ~ size", pd.concat(df_fec, axis=1), family='bernoulli')
    trace_fec = model_fec.fit(draws=5000, tune=10000, discard_tuned_samples=True, chains=1, progressbar=False)
    pickle.dump(trace_fec, open(file = os.getcwd()+f"/true_popu/gp_mle_true_population/MCMC/glm/sep/trace_fec.pkl", mode="wb"))

    # flow
    df_flow = XY_flow_compu(gp_mle_true_population)
    model_flow = bmb.Model("flow ~ size", pd.concat(df_flow, axis=1), family='poisson')
    trace_flow = model_flow.fit(draws=5000, tune=10000, discard_tuned_samples=True, chains=1, progressbar=False)
    pickle.dump(trace_flow, open(file = os.getcwd()+f"/true_popu/gp_mle_true_population/MCMC/glm/sep/trace_flow.pkl", mode="wb"))

    # for the 2nd datastes gp_mle_true_population with non-sep setting (seprate models for breeders and non-breeders).
    # grw, there is no needs to calculate the otehr vital rates.
    df_grw = XY_grw_compu(gp_mle_true_population)
    model_grw = bmb.Model("sizeNext ~ size", pd.concat(df_grw, axis=1))
    trace_grw = model_grw.fit(draws=5000, tune=10000, discard_tuned_samples=True, chains=1, progressbar=False)
    pickle.dump(trace_grw, open(file = os.getcwd()+f"/true_popu/gp_mle_true_population/MCMC/glm/nonsep/trace_grw.pkl", mode="wb"))

Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Sequential sampling (1 chains in 1 job)
NUTS: [sizeNext_sigma, Intercept, size]
Sampling 1 chain for 10_000 tune and 5_000 draw iterations (10_000 + 5_000 draws total) took 4 seconds.
Only one chain was sampled, this makes it impossible to run some convergence checks
Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Sequential sampling (1 chains in 1 job)
NUTS: [sizeNext_sigma, Intercept, size]
Sampling 1 chain for 10_000 tune and 5_000 draw iterations (10_000 + 5_000 draws total) took 4 seconds.
Only one chain was sampled, this makes it impossible to run some convergence checks
Modeling the probability that sur==1
Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Sequential sampling (1 chains in 1 job)
NUTS: [Intercept, size]
Sampling 1 chain for 10_000 tune and 5_000 draw iterations (10_000 + 5_000 draws total) took 4 seconds.
Only one chain was sampled, thi